<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue is a human-review prioritization list, not an automated publishing or content-decision system.

Each content item receives a priority based on its observed signals and is assigned a reason code that explains why it appears in the queue. The reason code should be understandable without inspecting the model.

### Action mapping

| Reason code | Action | Interpretation |
|---|---|---|
| STALE_HIGH_VISIBILITY | REFRESH | Content is relatively stale while still showing substantial observed search visibility. |
| STALE_LOW_VISIBILITY | REVIEW | Content is stale but has limited observed visibility; review before investing in a refresh. |
| FRESH_DECLINING | INVESTIGATE | Content is relatively recent but shows a decline signal; investigate before recommending a refresh. |
| LOW_VISIBILITY | REVIEW | Content has limited observed search visibility and needs human assessment before action. |
| OTHER | REVIEW | The item does not match a stronger rule-based reason and requires human assessment. |

The queue is intended to answer **"which pages should a content team review first?"**, not **"which pages must be changed?"**

Reason codes provide an auditable explanation for each recommendation. They should be treated as decision-support signals rather than causal explanations.

In [2]:
# Section 1 — Build the ranked action queue

import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Basic observed signals used for prioritization
df["stale_90_plus"] = df["days_since_last_update"] >= 90

# Visibility thresholds follow the exploratory buckets used earlier
high_visibility = df["impressions_90d"] >= 10000
low_visibility = df["impressions_90d"] < 100

# Build transparent reason codes
df["reason_code"] = np.select(
    [
        df["stale_90_plus"] & high_visibility,
        df["stale_90_plus"] & ~high_visibility,
        ~df["stale_90_plus"] & (df["trend_direction"] == "down"),
        low_visibility
    ],
    [
        "STALE_HIGH_VISIBILITY",
        "STALE_LOW_VISIBILITY",
        "FRESH_DECLINING",
        "LOW_VISIBILITY"
    ],
    default="OTHER"
)

# Human-readable action labels
df["action_label"] = df["reason_code"].map({
    "STALE_HIGH_VISIBILITY": "REFRESH",
    "STALE_LOW_VISIBILITY": "REVIEW",
    "FRESH_DECLINING": "INVESTIGATE",
    "LOW_VISIBILITY": "REVIEW",
    "OTHER": "REVIEW"
})

# Transparent priority score.
# This is a queue score, not a probability or causal score.
df["action_score"] = (
    df["stale_90_plus"].astype(int)
    + high_visibility.astype(int)
    + (df["trend_direction"] == "down").astype(int)
)

# Add concise explanations
df["why_its_here"] = df["reason_code"].map({
    "STALE_HIGH_VISIBILITY":
        "Observed content staleness combined with substantial observed search visibility.",
    "STALE_LOW_VISIBILITY":
        "Observed content staleness with relatively limited observed search visibility.",
    "FRESH_DECLINING":
        "The content is relatively recent but has an observed declining trend signal.",
    "LOW_VISIBILITY":
        "Observed search visibility is low, so human review is needed before prioritizing investment.",
    "OTHER":
        "No stronger action rule was triggered; human review is required."
})

df["what_would_make_it_wrong"] = df["reason_code"].map({
    "STALE_HIGH_VISIBILITY":
        "The page may already be performing adequately, or the observed metrics may not represent current performance.",
    "STALE_LOW_VISIBILITY":
        "Low visibility may reflect limited demand rather than a content-quality problem.",
    "FRESH_DECLINING":
        "The observed decline may be temporary or driven by factors unrelated to content quality.",
    "LOW_VISIBILITY":
        "Low visibility alone does not establish that refreshing the page will improve performance.",
    "OTHER":
        "The available signals are insufficient to justify a stronger recommendation."
})

queue_cols = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "action_score",
    "reason_code",
    "action_label",
    "why_its_here",
    "what_would_make_it_wrong"
]

action_queue = (
    df[queue_cols]
    .sort_values(
        ["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("Ranked action queue created.")
print("Rows:", len(action_queue))

display(action_queue.head(10))

Ranked action queue created.
Rows: 30000


,content_id,client_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label,why_its_here,what_would_make_it_wrong
0,content_5fe46e04994d,client_4e07408562,104,517715,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
1,content_2c2606c5d176,client_19581e27de,104,347399,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
2,content_cb112fce36be,client_19581e27de,104,309910,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
3,content_9532f197bbc8,client_4e07408562,104,309192,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
4,content_813e88069237,client_6208ef0f77,104,233561,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
5,content_c8e9d6ab9013,client_19581e27de,104,208678,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
6,content_3d94572c3a35,client_19581e27de,104,190623,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
7,content_01908772c6db,client_19581e27de,104,187893,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
8,content_05e9b4cd9ccf,client_6208ef0f77,104,179002,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
9,content_89fcb6f35525,client_6208ef0f77,104,174408,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help a content team prioritize which pages to review first.

The ranked queue combines observed content staleness, observed search visibility, and the observed decline signal into transparent action categories. The output is decision-support: it helps a reviewer decide where to spend attention first.

The queue is not intended to automatically publish, rewrite, delete, redirect, or otherwise change content.

### Limits

The recommendations are based on the available snapshot and observed signals. They should not be interpreted as evidence that taking the recommended action will cause better performance.

The Week-6 validation audit found that measured model performance depends on the validation design. It also identified a potential overlap between some 90-day performance features and the outcome window used to construct the decline label.

Therefore, the queue should be treated as a human-reviewed prioritization tool rather than a deployment-ready autonomous system.

The queue should also be reconsidered when the underlying data becomes stale, measurement definitions change, client populations change substantially, or the content strategy changes.

A high-priority page is not automatically a page that should be refreshed. It is a page that appears worth reviewing first.

In [3]:
# Section 2 — Intended use and limits

print("INTENDED USE / LIMITS CHECK")
print("=" * 50)

print("\nQueue rows:", len(action_queue))
print("Unique clients:", action_queue["client_id"].nunique())

print("\nAction distribution:")
display(
    action_queue["action_label"]
    .value_counts()
    .rename_axis("action_label")
    .reset_index(name="n")
)

print("\nReason-code distribution:")
display(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

print("\nHighest action score:", action_queue["action_score"].max())
print("Lowest action score:", action_queue["action_score"].min())

print("\nInterpretation:")
print(
    "The queue is a prioritization aid for human review, "
    "not an autonomous content-action system."
)

INTENDED USE / LIMITS CHECK

Queue rows: 30000
Unique clients: 32

Action distribution:


,action_label,n
0,REVIEW,17790
1,INVESTIGATE,10576
2,REFRESH,1634



Reason-code distribution:


,reason_code,n
0,FRESH_DECLINING,10576
1,STALE_LOW_VISIBILITY,7711
2,OTHER,5793
3,LOW_VISIBILITY,4286
4,STALE_HIGH_VISIBILITY,1634



Highest action score: 3
Lowest action score: 0

Interpretation:
The queue is a prioritization aid for human review, not an autonomous content-action system.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every queued recommendation requires human review before any content change.

### Human review checklist

Before acting on a recommendation, the reviewer should check:

1. Whether the page is still relevant to the current content strategy.
2. Whether the observed search visibility and trend signals are representative of the page's current situation.
3. Whether the page has important business, legal, brand, or editorial context that is not represented in the dataset.
4. Whether the recommended action is appropriate for the search intent and purpose of the page.
5. Whether the page has recently been changed or is already being worked on.
6. Whether the expected effort and potential value justify prioritizing the page.

### No-go list

The system should NOT automatically:

- publish or rewrite content;
- delete, redirect, or deindex a page;
- change titles, claims, or factual statements;
- make legal, medical, financial, or brand-sensitive decisions;
- treat a high action score as proof that a refresh will improve performance;
- override an experienced content reviewer;
- make decisions using information that was not available when the underlying signals were measured.

The queue identifies pages that appear worth reviewing first. It does not determine the final action.

In [4]:
review_requirements = [
    "Check current page relevance",
    "Verify observed performance signals",
    "Check recent content changes",
    "Review search intent and page purpose",
    "Check business/editorial/legal/brand context",
    "Assess effort versus potential value"
]

no_go_actions = [
    "Automatic publishing or rewriting",
    "Automatic deletion or redirection",
    "Automatic changes to factual or sensitive claims",
    "Automatic legal/medical/financial decisions",
    "Treating action score as causal evidence",
    "Overriding human reviewers"
]

print("HUMAN REVIEW / NO-GO CHECK")
print("=" * 50)

print("\nHuman review requirements:", len(review_requirements))
for i, item in enumerate(review_requirements, 1):
    print(f"{i}. {item}")

print("\nNo-go automated actions:", len(no_go_actions))
for i, item in enumerate(no_go_actions, 1):
    print(f"{i}. {item}")

print("\nFinal rule:")
print(
    "No content action should be executed solely from the queue. "
    "A human reviewer must make the final decision."
)

HUMAN REVIEW / NO-GO CHECK

Human review requirements: 6
1. Check current page relevance
2. Verify observed performance signals
3. Check recent content changes
4. Review search intent and page purpose
5. Check business/editorial/legal/brand context
6. Assess effort versus potential value

No-go automated actions: 6
1. Automatic publishing or rewriting
2. Automatic deletion or redirection
3. Automatic changes to factual or sensitive claims
4. Automatic legal/medical/financial decisions
5. Treating action score as causal evidence
6. Overriding human reviewers

Final rule:
No content action should be executed solely from the queue. A human reviewer must make the final decision.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should be monitored for changes in data quality, population, recommendation mix, and observed performance.

### Monitoring checks

Monitor:

- missing-value rates for the signals used by the queue;
- distribution of `days_since_last_update` and `impressions_90d`;
- the proportion of pages receiving each action label;
- changes in the client population;
- whether reason-code frequencies change substantially;
- whether reviewed recommendations remain useful to the content team.

### Retrain / review triggers

The model or prioritization rules should be reviewed when:

1. the underlying feature definitions change;
2. the client or content population changes materially;
3. missingness or data-quality problems increase;
4. the distribution of key input features shifts substantially;
5. the action distribution changes unexpectedly;
6. observed out-of-sample ranking performance deteriorates on newly reviewed data.

These are review triggers, not automatic retraining commands. A human should inspect the cause of the change before deciding whether the model or rules need to be updated.

Because the current evidence is based on an observed snapshot and the Week-6 audit identified feature-window limitations, monitoring should focus on whether the same signals and validation assumptions remain appropriate over time.

In [5]:
# Section 4 — Monitoring / retrain triggers

print("MONITORING / RETRAIN CHECK")
print("=" * 50)

print("\n1. Current missingness:")
monitor_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

display(
    df[monitor_features]
    .isna()
    .sum()
    .rename("missing_n")
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\n2. Current action mix:")
display(
    action_queue["action_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percent")
    .reset_index()
    .rename(columns={"index": "action_label"})
)

print("\n3. Current reason-code mix:")
display(
    action_queue["reason_code"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percent")
    .reset_index()
    .rename(columns={"index": "reason_code"})
)

print("\n4. Monitoring rule:")
print(
    "Unexpected changes in data quality, population, feature distributions, "
    "action mix, or observed out-of-sample performance should trigger human review."
)

print("\n5. Retraining rule:")
print(
    "Retraining is not automatic. Investigate the cause of drift or "
    "performance deterioration before changing the model."
)

MONITORING / RETRAIN CHECK

1. Current missingness:


,feature,missing_n
0,days_since_last_update,0
1,impressions_90d,0
2,ctr,0
3,avg_position,0



2. Current action mix:


,action_label,percent
0,REVIEW,59.30
1,INVESTIGATE,35.25
2,REFRESH,5.45



3. Current reason-code mix:


,reason_code,percent
0,FRESH_DECLINING,35.25
1,STALE_LOW_VISIBILITY,25.70
2,OTHER,19.31
3,LOW_VISIBILITY,14.29
4,STALE_HIGH_VISIBILITY,5.45



4. Monitoring rule:
Unexpected changes in data quality, population, feature distributions, action mix, or observed out-of-sample performance should trigger human review.

5. Retraining rule:
Retraining is not automatic. Investigate the cause of drift or performance deterioration before changing the model.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported as a CSV so that the paper can reuse the same decision-support output produced by this notebook.

The queue is regenerated by the notebook rather than treated as a manually edited data artifact. The exported queue contains the ranked content items, reason codes, action labels, and explanations needed to support the recommendations section.

The export is an analytical output, not a production decision system. It should be regenerated when the underlying data or playbook rules change.

No automated content changes are performed by this notebook.

In [6]:
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "ranked_action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

print("Export complete.")
print("Queue file:", queue_path)
print("Rows exported:", len(action_queue))
print("Columns exported:", len(action_queue.columns))

print("\nExported columns:")
print(list(action_queue.columns))

Export complete.
Queue file: work/outputs/ranked_action_queue.csv
Rows exported: 30000
Columns exported: 9

Exported columns:
['content_id', 'client_id', 'days_since_last_update', 'impressions_90d', 'action_score', 'reason_code', 'action_label', 'why_its_here', 'what_would_make_it_wrong']


In [7]:
# Verify the exported queue

exported_queue = pd.read_csv(queue_path)

print("Export verification")
print("=" * 50)

print("Rows:", len(exported_queue))
print("Columns:", len(exported_queue.columns))

print(
    "Content IDs preserved:",
    exported_queue["content_id"].nunique() == action_queue["content_id"].nunique()
)

print(
    "Reason codes present:",
    exported_queue["reason_code"].notna().all()
)

print(
    "Action labels present:",
    exported_queue["action_label"].notna().all()
)

display(exported_queue.head(5))

Export verification
Rows: 30000
Columns: 9
Content IDs preserved: True
Reason codes present: True
Action labels present: True


,content_id,client_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label,why_its_here,what_would_make_it_wrong
0,content_5fe46e04994d,client_4e07408562,104,517715,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
1,content_2c2606c5d176,client_19581e27de,104,347399,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
2,content_cb112fce36be,client_19581e27de,104,309910,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
3,content_9532f197bbc8,client_4e07408562,104,309192,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."
4,content_813e88069237,client_6208ef0f77,104,233561,3,STALE_HIGH_VISIBILITY,REFRESH,Observed content staleness combined with subst...,"The page may already be performing adequately,..."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.